In [2]:
# ======================================
# STEP 1: Install Libraries
# ======================================
pip install langchain-openai gradio

SyntaxError: invalid syntax (39039365.py, line 4)

In [5]:
pip show gradio

Name: gradio
Version: 6.10.0
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License-Expression: Apache-2.0
Location: c:\Users\admin\anaconda3\Lib\site-packages
Requires: aiofiles, anyio, audioop-lts, brotli, fastapi, ffmpy, gradio-client, groovy, hf-gradio, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pytz, pyyaml, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 
Note: you may

In [6]:
# ======================================
# STEP 1: Load ENV
# ======================================
import os
from dotenv import load_dotenv

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")


# ======================================
# STEP 2: NVIDIA LLM
# ======================================
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1",
    model="meta/llama3-70b-instruct",
    temperature=0
)


# ======================================
# STEP 3: Dummy Bank DB
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: MCP Tools
# ======================================
def get_balance(account_id):

    if account_id in accounts:

        return f"💰 Balance: ₹{accounts[account_id]['balance']}"

    return "❌ Account not found"



def approve_loan(account_id):

    if account_id in accounts:

        return f"🏦 Loan approved for {account_id}"

    return "❌ Account not found"


# ======================================
# STEP 5: RBAC Security
# ======================================
def secure_access(role, user_account, requested_account, action):

    if role == "manager":

        return True

    elif role == "employee":

        return action != "approve_loan"

    elif role == "customer":

        return (
            user_account == requested_account
            and action != "approve_loan"
        )

    return False


# ======================================
# STEP 6: Intent detection via LLM
# ======================================
def decide_action(query):

    prompt = f"""
    Classify into ONE action:

    get_balance
    approve_loan

    Query: {query}

    Return only action name.
    """

    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    return response.content.strip().lower()


# ======================================
# STEP 7: MCP Agent
# ======================================
def banking_agent(
    message,
    role,
    user_account,
    requested_account,
    history
):

    if history is None:
        history = []

    if not user_account:

        reply = "⚠️ enter account id"

        history.append({
            "role": "assistant",
            "content": reply
        })

        return "", history


    if not requested_account:

        requested_account = user_account


    action = decide_action(message)


    if not secure_access(

        role,
        user_account,
        requested_account,
        action
    ):

        reply = "🚫 Access denied"

    else:

        if action == "get_balance":

            reply = get_balance(requested_account)

        elif action == "approve_loan":

            reply = approve_loan(requested_account)

        else:

            reply = "🤖 ask about balance or loan"


    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": reply
    })


    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr


with gr.Blocks() as demo:

    gr.Markdown("## 🏦 Banking MCP + NVIDIA")


    role = gr.Dropdown(

        ["customer", "employee", "manager"],

        value="customer",

        label="Role"
    )


    user_account = gr.Textbox(

        value="1001",

        label="Your Account ID"
    )


    requested_account = gr.Textbox(

        label="Target Account (optional)"
    )


    chatbot = gr.Chatbot(height=400)


    msg = gr.Textbox(label="Ask question")


    state = gr.State([])


    msg.submit(

        banking_agent,

        inputs=[

            msg,
            role,
            user_account,
            requested_account,
            state

        ],

        outputs=[msg, chatbot]

    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
